# Perturbation arms — leak & robustness (Tier-1 A / B / C)

Three evaluation arms, one shared mechanism: transform the render-time object between
`build_inputs` and the model call, re-run, and compare to the clean baseline. The arms
live in `src.layered.perturb`; the comparison metrics in
`evaluation.perturbation_bench`.

| arm | paper | question | good direction |
|---|---|---|---|
| **A** input-perturbation | Canayaz | does a content flip flip the call? | large response = reading evidence |
| **B** scrambled-prior | Han et al. | does mislabeling the panel move the PM? | large response = reading evidence |
| **C** permutation battery | Homo Silicus | is IC stable across meaning-preserving variants? | small spread = robust |

**Honesty.** These never feed a shipped run — an arm is selected by an explicit
`--perturb` flag, recorded in `.meta.json`, and (analyst side) gated by
`board.IDENTITY_KEYS` so a board can't mix perturbed and clean legs.

**Reproducibility.** Transforms are deterministic; a real run still carries the model's
own sampling noise (the client has no seed/temperature). The offline cells below use no
model at all.

## Inspect a perturbed prompt — no spend

```bash
# arm A: the change/momentum measurements flip; the level is held
python3 -m src.run_analyst --driver inflation --dry-run --asof 2023-08-01 \
        --perturb signflip_momentum

# arm C: meaning-preserving surface change
python3 -m src.run_analyst --driver inflation --dry-run --perturb reword_scaffold

# arm B: the PM brief with reports rotated under the wrong driver labels
python3 -m src.run_pm_ic --pod duration --dry-run --scramble-reports
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.data.fred_local import load_bundle
from src.layered.analysts.llm_analyst import LLMAnalyst
from src.layered.perturb import SignFlipMomentum, analyst_perturbation, ANALYST_NAMES
from src.layered.timeline import AsOf
print('analyst arms:', ANALYST_NAMES)

macro = load_bundle(['CPIAUCSL', 'PCEPILFE'])
world = AsOf(asof=pd.Timestamp('2023-08-01'), macro=macro, prices=pd.DataFrame())
clean   = LLMAnalyst.from_persona('inflation', text_selector=None)
flipped = LLMAnalyst.from_persona('inflation', text_selector=None, perturbation=SignFlipMomentum())
fc, tc = clean.build_inputs(world);  ff, tf = flipped.build_inputs(world)
print('level held:', fc.level, '->', ff.level)
for a, b in zip(clean._user_prompt(fc, tc).splitlines(), flipped._user_prompt(ff, tf).splitlines()):
    if a != b:
        print('  base :', a)
        print('  flip :', b)

## The PM scramble (arm B) — offline

Rotate each present driver's view into another slot, so the header (the prior) disagrees
with the report body (the evidence). Driver *keys* are untouched, so grounding and the
submit enum still see the real driver set.

In [ ]:
from src.layered.pm.board import ViewBoard
from src.layered.pm.brief import render_brief
from src.layered.perturb.brief import ScrambleReports

board = ViewBoard.from_dir('../reports/ab', '_on', check_identity=False)
m = board.at(board.meeting_dates(freq='ME')[-1])
scr = ScrambleReports().apply_meeting(m)
print('same driver set :', set(scr.present) == set(m.present))
print('brief changed   :', render_brief(scr) != render_brief(m))
print('original intact :', all(m.entries[d].view.driver == d for d in m.present))

## Scoring a perturbed run against its baseline

Once the scored arms have run (they cost API calls — see below), compare them.

- **A** `direction_response(baseline, perturbed)` — `flip_rate`: fraction of non-flat
  calls whose sign reversed. **High = reading the evidence; low = the recall fingerprint.**
- **C** `ic_stability({label: path, ...})` + `ic_dispersion(table)` — **small spread = robust.**
- **B** `scramble_response(baseline_pm, scrambled_pm)` — per-driver sign-change rate.
  **High = the PM read the (mislabeled) evidence; low = it recited the label's prior.**

```bash
# the scored arms (real spend). Register them under their own out paths:
python3 -m src.run_analyst_ic --driver inflation --perturb signflip_momentum \
        --model claude-haiku-4-5-20251001 --limit 24 \
        --out reports/perturb/inflation_signflip.jsonl
python3 -m src.run_pm_ic --pod duration --scramble-reports \
        --out reports/pm/duration_scramble.jsonl
```

In [ ]:
from src.layered.evaluation import direction_response, ic_stability, ic_dispersion, scramble_response

# Example wiring against whatever perturbed runs exist. Baseline-vs-itself is the
# identity check: flip_rate 0, dispersion 0.
base = '../reports/ab/inflation_on.jsonl'
print('A self-check:', direction_response(base, base))
tab = ic_stability({'baseline': base})
print('C table:'); print(tab)
print('C dispersion:', ic_dispersion(tab))